# 🧰 Skill Mode — Alteryx → Databricks Converter

**End-to-end conversion of Alteryx `.yxmd` workflows into validated, Databricks-ready PySpark notebooks.**

This is the *only* notebook you need. All heavy logic lives in the `src/` Python modules of this
[Databricks Asset Bundle](https://docs.databricks.com/dev-tools/bundles/index.html); this notebook is a thin orchestration layer.

| Section | What it does |
|---|---|
| 1 | Setup & install dependencies |
| 2 | Upload / configure Alteryx workflow(s) |
| 3 | Parse & visualize the workflow DAG |
| 4 | Configure conversion settings |
| 5 | Execute conversion |
| 6 | Self-correction & validation loop (up to 3 iterations) |
| 7 | Download / export the Databricks notebook (`.ipynb`, `.py`, `.dbc`) |
| 8 | Bundle deploy (`databricks.yml` + resources) |
| 9 | (Optional) validate output against sample data |

> Runs on **Databricks Runtime 13.3+** or local **PySpark 3.4+ / Python 3.9+**.

## 1️⃣ Setup & Install Dependencies

In [ ]:
# On Databricks this installs into the notebook-scoped environment.
# Locally: pip install -r requirements.txt
%pip install -q lxml pyyaml nbformat

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

# Locate the bundle root so `src/` and `config/` are importable whether this
# notebook runs from a Databricks bundle deployment, a Git folder, or locally.
def _find_bundle_root():
    candidates = [Path.cwd()] + list(Path.cwd().parents)
    try:  # Databricks: notebook lives in <root>/notebooks/
        candidates.insert(0, Path(
            dbutils.notebook.entry_point.getDbutils().notebook().getContext()
            .notebookPath().get()).parent.parent)
    except NameError:
        pass
    for base in candidates:
        if (base / "src" / "converter_engine.py").exists():
            return base
        if (base / "notebooks").exists() and (base.parent / "src").exists():
            return base.parent
    return Path.cwd()

BUNDLE_ROOT = _find_bundle_root()
sys.path.insert(0, str(BUNDLE_ROOT))
print(f"Bundle root: {BUNDLE_ROOT}")

# Route logs to the Databricks driver log (stderr) — no print-based logging.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s [%(name)s] %(message)s",
    stream=sys.stderr,
)
logger = logging.getLogger("skill_mode")

from src.parser import AlteryxWorkflowParser
from src.converter_engine import ConverterEngine, load_tool_mapping
from src.self_correction import SelfCorrectingConverter
from src.databricks_exporter import DatabricksExporter, batch_export, safe_name
from src.validators import StructuralValidator, run_generated_code

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
print(f"Running on Databricks: {IS_DATABRICKS}")

## 2️⃣ Upload / Configure Alteryx Workflow(s)

Three ways to provide input — pick one:

1. **Widget path** — point `yxmd_path` at a `.yxmd` file (Workspace file, Volume, or repo path).
2. **Paste XML** — paste workflow XML into `PASTED_XML` below.
3. **Batch mode** — set `batch_mode = true` and point `input_dir` at a folder of `.yxmd` files.

In [ ]:
# Configuration widgets (dbutils on Databricks, env-var/default fallback locally)
WIDGETS = {
    "yxmd_path":       str(BUNDLE_ROOT / "tests/sample_workflows/02_join_summarize.yxmd"),
    "batch_mode":      "false",
    "input_dir":       str(BUNDLE_ROOT / "tests/sample_workflows"),
    "output_dir":      str(BUNDLE_ROOT / "output"),
    "output_format":   "ipynb,py",       # any of: ipynb, py, dbc
    "target_catalog":  "main",           # Unity Catalog target for output tables
    "target_schema":   "alteryx_migrated",
    "bundle_target":   "dev",            # databricks.yml target for section 8
    "max_iterations":  "3",              # self-correction budget
    "debug_mode":      "false",          # print schemas/samples per converted tool
    "run_sample_validation": "false",    # execute generated code on this cluster
}

def get_widget(name):
    if IS_DATABRICKS:
        try:
            dbutils.widgets.text(name, WIDGETS[name], name)
            return dbutils.widgets.get(name)
        except Exception:
            pass
    return os.environ.get(f"SKILL_{name.upper()}", WIDGETS[name])

CFG = {name: get_widget(name) for name in WIDGETS}
CFG["batch_mode"] = CFG["batch_mode"].strip().lower() == "true"
CFG["debug_mode"] = CFG["debug_mode"].strip().lower() == "true"
CFG["run_sample_validation"] = CFG["run_sample_validation"].strip().lower() == "true"
CFG["max_iterations"] = int(CFG["max_iterations"])
CFG["formats"] = tuple(f.strip() for f in CFG["output_format"].split(",") if f.strip())
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

In [ ]:
# Option 2: paste raw .yxmd XML here (leave empty to use yxmd_path / batch mode)
PASTED_XML = """""".strip()

import tempfile
if PASTED_XML:
    pasted_file = Path(tempfile.gettempdir()) / "pasted_workflow.yxmd"
    pasted_file.write_text(PASTED_XML, encoding="utf-8")
    CFG["yxmd_path"] = str(pasted_file)
    CFG["batch_mode"] = False
    print(f"Using pasted XML → {pasted_file}")

workflow_files = (
    sorted(Path(CFG["input_dir"]).rglob("*.yxmd")) if CFG["batch_mode"]
    else [Path(CFG["yxmd_path"])]
)
for f in workflow_files:
    assert f.exists(), f"Workflow file not found: {f}"
print(f"{len(workflow_files)} workflow(s) queued:")
for f in workflow_files:
    print(f"  • {f.name}")

## 3️⃣ Parse & Visualize the Workflow DAG

In [ ]:
def render_dag(workflow, title):
    """Render the tool DAG as an HTML table + edge list (Databricks-native)."""
    rows = []
    for tid in sorted(workflow.all_tools):
        tool = workflow.all_tools[tid]
        inputs = [c.origin_tool_id for c in workflow.get_incoming_connections(tid)]
        rows.append(
            f"<tr><td>{tid}</td><td><b>{tool.tool_type}</b></td>"
            f"<td>{tool.annotation or '—'}</td>"
            f"<td>{', '.join(map(str, inputs)) or 'source'}</td></tr>")
    html = (
        f"<h3>🗺️ {title}</h3>"
        f"<p>{len(workflow.all_tools)} tools · {len(workflow.connections)} connections"
        f" · {len(workflow.containers)} containers</p>"
        "<table border='1' cellpadding='4' style='border-collapse:collapse'>"
        "<tr><th>Tool ID</th><th>Type</th><th>Annotation</th><th>Feeds from</th></tr>"
        + "".join(rows) + "</table>")
    try:
        displayHTML(html)  # noqa: F821 — Databricks builtin
    except NameError:
        for tid in sorted(workflow.all_tools):
            tool = workflow.all_tools[tid]
            ins = [c.origin_tool_id for c in workflow.get_incoming_connections(tid)]
            print(f"  [{tid:>4}] {tool.tool_type:<18} ← {ins or 'source'}  {tool.annotation}")

parsed_workflows = {}
for f in workflow_files:
    wf = AlteryxWorkflowParser(str(f)).parse()
    parsed_workflows[f.stem] = wf
    render_dag(wf, f.name)

## 4️⃣ Configure Conversion Settings

The engine is driven by [`config/tool_mapping.yaml`](../config/tool_mapping.yaml) — every Alteryx tool
maps to a `ToolConverter` plugin in `src/tools/`. Override input-table locations below so Alteryx
data sources resolve to Unity Catalog tables (`catalog.schema.table`).

In [ ]:
tool_mapping = load_tool_mapping(str(BUNDLE_ROOT / "config" / "tool_mapping.yaml"))
print(f"Tool mapping v{tool_mapping.get('version')}: "
      f"{len(tool_mapping.get('tools', {}))} tool types registered")

# Map Alteryx input tools → Unity Catalog tables.
# Keys match tool ID, annotation substring, or source path substring.
# Example: {"12": "main.bronze.channel", "transactions": "main.sales.transactions"}
SOURCE_TABLES = {}

source_tables_file = BUNDLE_ROOT / "config" / "source_tables.json"
if source_tables_file.exists():
    SOURCE_TABLES.update(json.loads(source_tables_file.read_text()))
    print(f"Loaded {len(SOURCE_TABLES)} source-table mappings from {source_tables_file.name}")

engine = ConverterEngine(
    tool_mapping_path=str(BUNDLE_ROOT / "config" / "tool_mapping.yaml"),
    source_tables_config=SOURCE_TABLES,
    target_catalog=CFG["target_catalog"],
    target_schema=CFG["target_schema"],
)
converter = SelfCorrectingConverter(engine=engine, max_iterations=CFG["max_iterations"])
print("Converter ready.")

## 5️⃣ Execute Conversion & 6️⃣ Self-Correction Loop

Each workflow goes through **generate → validate → correct**:

- **Validate**: the generated PySpark is parsed with Python's `ast` module and compared against
  the original `.yxmd` DAG — tool coverage, join keys, filter fields, unresolved references, syntax.
- **FAILED** → regenerate in *strict mode* (granular re-parse of each tool's raw XML config).
- **WARNING** → apply optimization rules (broadcast joins, Delta writes, ZORDER hints) without touching logic.
- Up to `max_iterations` (default 3) passes; if still failing, the best attempt ships with a
  detailed error report as the notebook's top `%md` cell.

In [ ]:
def show_progress(rows):
    """Live progress tracker: one row per workflow/iteration."""
    body = "".join(
        f"<tr><td>{w}</td><td>{i}</td><td>{s}</td><td>{m}</td></tr>" for w, i, s, m in rows)
    html = ("<table border='1' cellpadding='4' style='border-collapse:collapse'>"
            "<tr><th>Workflow</th><th>Iteration</th><th>Status</th><th>Action</th></tr>"
            + body + "</table>")
    try:
        displayHTML(html)  # noqa: F821
    except NameError:
        for w, i, s, m in rows:
            print(f"  {w} · iter {i} · {s} · {m}")

spark_session = None
if CFG["run_sample_validation"]:
    try:
        spark_session = spark  # noqa: F821 — Databricks builtin
    except NameError:
        try:
            from pyspark.sql import SparkSession
            spark_session = SparkSession.builder.appName("skill-mode-validate").getOrCreate()
        except ImportError:
            logger.warning("pyspark unavailable — skipping sample-data validation")

progress_rows, results = [], {}
for name, wf in parsed_workflows.items():
    logger.info("Converting %s", name)
    result = converter.convert(
        wf, name, spark=spark_session,
        progress_callback=lambda i, s, m, _n=name: progress_rows.append((_n, i, s, m)))
    results[name] = result

    if CFG["debug_mode"]:
        print(f"--- {name}: per-tool conversion status ---")
        for st in result.conversion.tool_statuses:
            print(f"  [{st.tool_id:>4}] {st.tool_type:<18} {st.status:<12} {st.message[:80]}")

show_progress(progress_rows)
for name, result in results.items():
    icon = {"PASS": "✅", "WARNING": "⚠️", "FAIL": "❌"}[result.status]
    print(f"{icon} {name}: {result.status} after {len(result.iterations)} iteration(s), "
          f"{result.conversion.num_tools} tools "
          f"({len(result.conversion.unsupported_tools)} unsupported)")

## 7️⃣ Download / Export the Databricks Notebook

The validated (or best-effort) notebook is exported with the **validation report as its first `%md`
cell**. Formats: `.ipynb` (import to any workspace / Git folder), `.py` (Databricks source format),
`.dbc` (workspace archive). In batch mode the output mirrors the input folder structure.

In [ ]:
exporter = DatabricksExporter(CFG["output_dir"])
exported = {}
for name, result in results.items():
    exported[name] = exporter.export(result, formats=CFG["formats"])
    for fmt, path in exported[name].items():
        print(f"📦 {name}.{fmt} → {path}")

summary_path = Path(CFG["output_dir"]) / "conversion_summary.json"
summary_path.parent.mkdir(parents=True, exist_ok=True)
summary_path.write_text(json.dumps({
    name: {
        "status": r.status,
        "iterations": len(r.iterations),
        "tools": r.conversion.num_tools,
        "unsupported": [f"{s.tool_id}:{s.tool_type}" for s in r.conversion.unsupported_tools],
        "files": exported[name],
    } for name, r in results.items()
}, indent=2))
print(f"\n📄 Summary → {summary_path}")

## 8️⃣ Bundle Deploy

This repo **is** a Databricks Asset Bundle (`databricks.yml` at the root, job resources under
`resources/`). Converted notebooks under `output/` are picked up by the
`alteryx_migration_job` workflow resource. Deploy from a terminal:

```bash
databricks bundle validate                # always validate first
databricks bundle deploy  -t dev          # or: staging / prod
databricks bundle run alteryx_migration_job -t dev
```

Or from the Databricks UI: **Workspace → Git folders → this repo → Deploy** (bundle UI).
The cell below stages the generated notebooks into the bundle sync path and prints the exact commands.

In [ ]:
bundle_yaml = BUNDLE_ROOT / "databricks.yml"
print(f"Bundle root config: {bundle_yaml} (exists: {bundle_yaml.exists()})")
print(f"Job resource:       {BUNDLE_ROOT / 'resources' / 'alteryx_migration_job.yml'}")
print()
print("Deploy the converted notebooks with:")
print(f"  cd {BUNDLE_ROOT}")
print("  databricks bundle validate")
print(f"  databricks bundle deploy -t {CFG['bundle_target']}")
print(f"  databricks bundle run alteryx_migration_job -t {CFG['bundle_target']}")

## 9️⃣ (Optional) Validate Output Against Sample Data

Executes each generated notebook's code on this cluster and reports row counts and schemas per
DataFrame. Writes in generated notebooks are commented out by default, so this is side-effect free.
Requires an active Spark session (`run_sample_validation = true`).

In [ ]:
if spark_session is None:
    print("Sample-data validation skipped (set run_sample_validation = true and attach a cluster).")
else:
    for name, result in results.items():
        outcome = run_generated_code(result.code, spark_session)
        icon = "✅" if outcome["ok"] else "❌"
        print(f"{icon} {name}: execution {'succeeded' if outcome['ok'] else 'FAILED'}")
        if not outcome["ok"]:
            print(f"   {outcome['error']}")
        for df_name, meta in sorted(outcome.get("dataframes", {}).items()):
            print(f"   • {df_name}: {meta['rows']} rows × {len(meta['columns'])} cols")
            if CFG["debug_mode"]:
                print(f"     columns: {meta['columns']}")

---
### 🔌 Adding support for a new Alteryx tool

1. Create a `ToolConverter` subclass in `src/tools/` and decorate it with `@register("MyTool")`.
2. (Optional) declare it in `config/tool_mapping.yaml` to document support level — or use the YAML
   alone to point at any importable class, **no core changes needed**.
3. Add a sample workflow under `tests/sample_workflows/` and run `pytest`.

See the [README](../README.md) for the full architecture and the Databricks Marketplace listing guide.